In [1]:
begin
    using Pkg
    dev_folder = joinpath(@__DIR__, "../Examples")
    Pkg.activate(dev_folder)
end
Threads.nthreads()

  Activating project at `~/Realizibility_index/BindingAndCatalysis.jl/Examples`


24

In [2]:
using Polyhedra
using GLMakie # for plotting, we use Makie backend, could also be GLMakie or WGLMakie...
using Revise
using BindingAndCatalysis # import the package

[ Info: Precompiling BindingAndCatalysis [b532a4b1-2c45-4c12-bcaf-8695372aa40c] (cache misses: include_dependency fsize change (2), incompatible header (4))


In [46]:
function rref(A::Matrix{Int})
    M = copy(A)
    m, n = size(M)
    pivots = Int[]
    row = 1

    for col in 1:n
        pivot = nothing
        for r in row:m
            if M[r, col] != 0
                pivot = r
                break
            end
        end
        pivot === nothing && continue

        M[row, :], M[pivot, :] = M[pivot, :], M[row, :]

        M[row, :] ./= M[row, col]

        for r in 1:m
            if r != row && M[r, col] != 0
                M[r, :] .-= M[r, col] .* M[row, :]
            end
        end

        push!(pivots, col)
        row += 1
        row > m && break
    end

    return M, pivots
end

function ker_ST1(S::Matrix{Int})
    A = Rational{Int}.(transpose(S))
    M, pivots = rref(A)
    m, n = size(M)

    freecols = setdiff(1:n, pivots)
    B = zeros(Rational{Int}, n, length(freecols))

    for (j, fc) in enumerate(freecols)
        x = zeros(Rational{Int}, n)
        x[fc] = 1
        for (i, pc) in enumerate(pivots)
            x[pc] = -M[i, fc]
        end
        B[:, j] = x
    end

    # convert each rational basis vector to a primitive integer vector
    Bout = zeros(Int, size(B))
    for j in 1:size(B, 2)
        v = B[:, j]

        dens = denominator.(v)
        L = foldl(lcm, dens; init=1)

        w = Int.(L .* v)

        g = foldl(gcd, abs.(w); init=0)
        g = g == 0 ? 1 : g

        Bout[:, j] = div.(w, g)
    end

    return Bout, pivots
end

ker_ST1 (generic function with 1 method)

In [49]:
S = Matrix{Int}(BindingAndCatalysis.sprand(30,30,0.1) .|> round)

30×30 Matrix{Int64}:
 0  0  1  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  1
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  1     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  1  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  1  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  1  0  0  1  0  1  0  0  0  0     0  0  0  0  0  0  0  1  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  1  0  0  1
 1  0  0  0  0  0  1  0  0  0  0  1  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  1  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     

In [50]:
a, b = ker_ST1(S)

([0 0 … 0 0; 1 0 … 0 0; … ; 0 0 … 0 1; 0 0 … 0 0], [1, 4, 6, 7, 8, 10, 11, 13, 14, 16, 17, 18, 24, 27, 28, 30])

In [51]:
c,d = ker_ST2(S)

(14, [0 0 0 0 0 0 0 0 1 0 0 0 0 0; -1 0 0 0 0 0 0 0 0 0 0 0 0 0; 0 -1 0 0 0 0 0 0 0 0 0 0 0 0; 0 0 0 0 0 0 0 0 0 1 0 0 0 0; 0 0 -1 0 0 0 0 0 0 0 0 0 0 0; 0 0 0 0 1 0 0 1 0 0 0 0 0 0; 0 0 0 0 0 0 0 0 0 1 0 0 0 0; 0 0 0 0 0 0 0 0 0 0 0 0 0 0; 0 0 0 -1 0 0 0 0 0 0 0 0 0 0; 0 0 0 0 0 0 0 1 0 0 0 0 0 0; 0 0 0 0 0 0 0 0 0 0 0 0 0 0; 0 0 0 0 -1 0 0 0 0 0 0 0 0 0; 0 0 0 0 0 0 0 -1 -1 0 0 0 0 0; 0 0 0 0 0 0 0 0 0 0 0 0 0 0; 0 0 0 0 0 -1 0 0 0 0 0 0 0 0; 0 0 0 0 0 0 0 0 0 -1 0 0 0 0; 0 0 0 0 0 0 0 1 0 0 0 0 0 0; 0 0 0 0 0 0 0 0 0 0 0 0 0 0; 0 0 0 0 0 0 -1 0 0 0 0 0 0 0; 0 0 0 0 0 0 0 -1 0 0 0 0 0 0; 0 0 0 0 0 0 0 0 -1 0 0 0 0 0; 0 0 0 0 0 0 0 0 0 -1 0 0 0 0; 0 0 0 0 0 0 0 0 0 0 -1 0 0 0; 0 0 0 0 0 0 0 0 0 0 0 0 0 0; 0 0 0 0 0 0 0 0 0 0 0 -1 0 0; 0 0 0 0 0 0 0 0 0 0 0 0 -1 0; 0 0 0 0 0 0 0 0 0 0 0 0 0 0; 0 0 0 0 0 0 0 0 0 0 0 0 0 0; 0 0 0 0 0 0 0 0 0 0 0 0 0 -1; 0 0 0 0 0 0 0 0 0 0 0 0 0 0])

In [69]:
ker_ST1(S)[1]' * S

14×30 Matrix{Int64}:
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     

In [71]:
using BenchmarkTools

@assert all(ker_ST1(S)[1]' * S .== Ref(0))

@btime ker_ST1(S)

  23.524 μs (350 allocations: 120.33 KiB)


([0 0 … 0 0; 1 0 … 0 0; … ; 0 0 … 0 1; 0 0 … 0 0], [1, 4, 6, 7, 8, 10, 11, 13, 14, 16, 17, 18, 24, 27, 28, 30])

In [74]:
@assert all(left_nullspace_integer(S)[1]' * S .== Ref(0))

@btime left_nullspace_integer(S)

  12.297 μs (156 allocations: 57.80 KiB)


([0 0 … 0 0; 1 0 … 0 0; … ; 0 0 … 0 1; 0 0 … 0 0], [1, 4, 6, 7, 8, 10, 11, 13, 14, 16, 17, 18, 24, 27, 28, 30])

In [ ]:
# Row-reduced echelon form over an arbitrary exact scalar type,
# e.g. Rational{Int}. Returns:
#   M          : RREF(A)
#   pivotcols  : pivot columns of A
function rref_exact(A::AbstractMatrix{T}) where {T<:Number}
    M = copy(A)
    m, n = size(M)
    pivotcols = Int[]
    row = 1

    @inbounds for col in 1:n
        row > m && break

        # find pivot row
        pivot = 0
        for r in row:m
            if M[r, col] != 0
                pivot = r
                break
            end
        end
        pivot == 0 && continue

        # swap rows manually (avoids slice allocations)
        if pivot != row
            for j in 1:n
                M[row, j], M[pivot, j] = M[pivot, j], M[row, j]
            end
        end

        # normalize pivot row
        piv = M[row, col]
        if piv != one(T)
            for j in col:n
                M[row, j] /= piv
            end
        end

        # eliminate other rows
        for r in 1:m
            r == row && continue
            c = M[r, col]
            c == 0 && continue
            for j in col:n
                M[r, j] -= c * M[row, j]
            end
        end

        push!(pivotcols, col)
        row += 1
    end

    return M, pivotcols
end


# Convert a rational vector to a primitive integer vector.
# Optionally flips sign so the first nonzero entry is positive.
function primitive_integer(v::AbstractVector{<:Rational})
    dens = denominator.(v)
    L = foldl(lcm, dens; init=1)

    w = Int.(L .* v)

    g = foldl(gcd, abs.(w); init=0)
    g = g == 0 ? 1 : g
    w = div.(w, g)

    # canonical sign choice: first nonzero entry positive
    for x in w
        if x != 0
            if x < 0
                w = .-w
            end
            break
        end
    end

    return w
end


# Left nullspace basis of S:
# finds B such that B' * S = 0
# Returns:
#   B          : columns are primitive integer basis vectors for left nullspace
#   pivotrows  : pivot row indices of S
function left_nullspace_integer(S::AbstractMatrix{Int})
    A = Rational{Int}.(transpose(S))   # nullspace of S'
    M, pivotrows = rref_exact(A)

    m, n = size(M)                     # n = number of rows of S
    ispivot = falses(n)
    for c in pivotrows
        ispivot[c] = true
    end
    freecols = findall(!, ispivot)

    B = Matrix{Int}(undef, n, length(freecols))

    @inbounds for (j, fc) in enumerate(freecols)
        x = zeros(Rational{Int}, n)
        x[fc] = 1
        for (i, pc) in enumerate(pivotrows)
            x[pc] = -M[i, fc]
        end
        B[:, j] = primitive_integer(x)
    end

    return B, pivotrows
end

left_nullspace_integer (generic function with 1 method)

In [60]:
a' *S

14×30 Matrix{Int64}:
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     

In [61]:
Matrix{Int}(d)' *S

14×30 Matrix{Int64}:
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     

In [ ]:
using AbstractAlgebra

[ Info: Precompiling IJuliaExt [556a872b-b57d-539f-ba60-70313fab1f9a] 

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [34]:
using AbstractAlgebra
function ker_ST2(S::Matrix{Int})
    A = matrix(ZZ, S')
    return nullspace(A)
end

ker_ST2 (generic function with 1 method)

In [11]:
P = [1 -1]

pi = [0 0 1;
       0 1 0]
H =[-1;1;0]

3-element Vector{Int64}:
 -1
  1
  0

In [11]:
P* pi *H

1-element Vector{Int64}:
 -1

In [12]:
model = let 
    L = [0 1 1; 1 0 1]
    N = [1 1 -1]
    Bnc(L=L,N=N)
end

----------Binding Network Summary:-------------
Number of species (n): 3
Number of conserved quantities (d): 2
Number of reactions (r): 1
L matrix: [0 1 1; 1 0 1]
N matrix: [1 1 -1]
Direction of binding reactions: forward
Catalysis involved: No
Regimes constructed: No
-----------------------------------------------

In [19]:
get_H(model,2)

3×3 SparseMatrixCSC{Float64, Int64} with 5 stored entries:
 -1.0  1.0  1.0
  1.0   ⋅    ⋅ 
   ⋅   1.0   ⋅ 

In [23]:
M1= get_M(model,2)

3×3 SparseMatrixCSC{Int64, Int64} with 5 stored entries:
 ⋅  1   ⋅
 ⋅  ⋅   1
 1  1  -1

In [24]:
M2 = get_M(model,4)

3×3 SparseMatrixCSC{Int64, Int64} with 5 stored entries:
 ⋅  ⋅   1
 ⋅  ⋅   1
 1  1  -1

In [22]:
a = inv([0 0 1 ;1 1 -1;0 -1 1])

3×3 Matrix{Float64}:
 -0.0  1.0   1.0
  1.0  0.0  -1.0
  1.0  0.0   0.0

In [25]:
M1 *a

3×3 Matrix{Float64}:
 1.0  0.0  -1.0
 1.0  0.0   0.0
 0.0  1.0   0.0

In [26]:
M2 *a

3×3 Matrix{Float64}:
 1.0  0.0  0.0
 1.0  0.0  0.0
 0.0  1.0  0.0